In [151]:
import pandas as pd
import os

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import datetime


In [115]:
path = 'data/train.csv'

df = pd.read_csv(path, index_col='id')

In [116]:
feature_columns = df.columns

In [117]:
df.sample()

,track_name,album_type,album_name,release_date,record_label,duration_sec,audio_dance,audio_energy,audio_key,audio_loudness,...,audio_instrumental,audio_live,audio_valence,audio_tempo,time_signature,artist_name,artist_genres,artist_followers,explicit_lyrics,music_hit
id,,,,,,,,,,,,,,,,,,,,,
171276,Arise,compilation,Red Hot Reggae,2017-01-30,Music Manager,235.317,0.745,0.631,7,-6.309,...,0.000277,0.19,0.814,153.875,4,Various Artists,NaN,2316620,False,0


In [118]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 350051 entries, 1 to 350051
Data columns (total 23 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   track_name          350047 non-null  str    
 1   album_type          350051 non-null  str    
 2   album_name          350050 non-null  str    
 3   release_date        350051 non-null  str    
 4   record_label        350013 non-null  str    
 5   duration_sec        350051 non-null  float64
 6   audio_dance         350051 non-null  float64
 7   audio_energy        350051 non-null  float64
 8   audio_key           350051 non-null  int64  
 9   audio_loudness      350051 non-null  float64
 10  audio_mode          350051 non-null  int64  
 11  audio_speech        350051 non-null  float64
 12  audio_acoustic      350051 non-null  float64
 13  audio_instrumental  350051 non-null  float64
 14  audio_live          350051 non-null  float64
 15  audio_valence       350051 non-null  float64


In [125]:
scaler = StandardScaler()

standard_columns = ['duration_sec', 'audio_dance', 'audio_energy', 'audio_loudness', 'audio_speech', 'audio_acoustic', 'audio_instrumental', 'audio_live', 'audio_valence', 'audio_tempo']

scaler.fit(df[standard_columns])

StandardScaler()

In [129]:
df[standard_columns] = scaler.transform(df[standard_columns])
print(df.sample(1))
print(df.columns)

                                              track_name   album_type  \
id                                                                      
85167  Does Fort Worth Ever Cross Your Mind - Karaoke...  compilation   

                                              album_name release_date  \
id                                                                      
85167  Artist Karaoke, Vol. 398 : Sing The Songs Of G...   2010-05-23   

                        record_label  duration_sec  audio_dance  audio_energy  \
id                                                                              
85167  Charttraxx Home Entertainment     -1.837174    21.154997     -13.87253   

       audio_key  audio_loudness  ...  audio_instrumental  audio_live  \
id                                ...                                   
85167          0        1.987566  ...            7.523233   -1.484028   

       audio_valence  audio_tempo  time_signature         artist_name  \
id                     

In [100]:
#drop_columns = ['artist_name', 'record_label']
#df.drop(columns=drop_columns)

album_type_mapping = {
    "album": 1,
    "single": 2,
    "compilation": 3
}

df["album_type"] = df["album_type"].map(album_type_mapping)

df["explicit_lyrics"] = df["explicit_lyrics"].map( {True: 1, False: 0})


                                               track_name  album_type  \
id                                                                      
350052                                  The Barnyard Song           1   
350053                             It'S All Right With Me           1   
350054                                      It'S Delovely           1   
350055                                  Begin The Beguine           1   
350056                         I'Ve Got You Under My Skin           1   
...                                                   ...         ...   
437560                    O Santa Notte: Canzone Perfetta           1   
437561         2023 Christmas: Go Tell It On The Mountain           1   
437562  Symphony No.23 In D Major K.181(162B): 1. Alle...           1   
437563  Symphony No.23 In D Major K.181(162B): 3. Pres...           1   
437564                            Invitation - Radio Edit           2   

                                               alb

In [147]:
train = pd.read_csv('data/train.csv')

In [ ]:
#scaler = StandardScaler()

#standard_columns = ['duration_sec', 'audio_dance', 'audio_energy', 'audio_loudness', 'audio_speech', 'audio_acoustic', 'audio_instrumental', 'audio_live', 'audio_valence', 'audio_tempo']

#scaler.fit(train[standard_columns])

StandardScaler()

In [ ]:
#train[standard_columns] = scaler.transform(train[standard_columns])

In [164]:
categorical_columns = [
    "track_name",
    "album_type",
    "album_name",
    "record_label",
    "artist_name",
    "artist_genres",
]

train["explicit_lyrics"] = train["explicit_lyrics"].map( {True: 1, False: 0})

dates = pd.to_datetime(
    train["release_date"],
    errors="coerce",
    format="mixed"
)

train["release_date"] = dates.map(
    lambda value: value.toordinal()
    if pd.notna(value)
    else float("nan")
)

for column in categorical_columns:
    train[column] = train[column].fillna("missing").astype(str)

In [165]:

X = train.drop(columns=['music_hit'])
y = train['music_hit']

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [2]:
from catboost import CatBoostClassifier

In [5]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    
)


In [ ]:
model.fit(X_train, y_train, 
        cat_features=categorical_columns,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=100,
        plot = True
        )
model.save_model()

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.8812436	best: 0.8812436 (0)	total: 131ms	remaining: 2m 11s
100:	test: 0.9638703	best: 0.9638703 (100)	total: 11.8s	remaining: 1m 44s
200:	test: 0.9663028	best: 0.9663028 (200)	total: 24.3s	remaining: 1m 36s
300:	test: 0.9672371	best: 0.9672371 (300)	total: 37.6s	remaining: 1m 27s
400:	test: 0.9678023	best: 0.9678023 (400)	total: 51.1s	remaining: 1m 16s
500:	test: 0.9682257	best: 0.9682257 (500)	total: 1m 4s	remaining: 1m 4s
600:	test: 0.9684571	best: 0.9684576 (596)	total: 1m 19s	remaining: 52.8s
700:	test: 0.9686724	best: 0.9686724 (700)	total: 1m 33s	remaining: 39.9s
800:	test: 0.9688304	best: 0.9688307 (799)	total: 1m 47s	remaining: 26.7s
900:	test: 0.9689463	best: 0.9689505 (896)	total: 2m 1s	remaining: 13.3s
999:	test: 0.9690685	best: 0.9690699 (998)	total: 2m 15s	remaining: 0us

bestTest = 0.969069915
bestIteration = 998

Shrink model to first 999 iterations.


CatBoostClassifier(depth=7, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [185]:
test = pd.read_csv('data/test.csv')

test["explicit_lyrics"] = test["explicit_lyrics"].map( {True: 1, False: 0})

dates = pd.to_datetime(
    train["release_date"],
    errors="coerce",
    format="mixed"
)

test["release_date"] = dates.map(
    lambda value: value.toordinal()
    if pd.notna(value)
    else float("nan")
)

for column in categorical_columns:
    test[column] = test[column].fillna("missing").astype(str)

In [197]:
import csv

X_test = test.loc[:, model.feature_names_]

with open("submission.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["id", "music_hit"])

    for i, track_id in enumerate(test["id"]):
        row = X_test.iloc[[i]]
        hit_probability = model.predict_proba(row)[0, 1]
        writer.writerow([track_id, hit_probability])